In [2]:
import torch
import xarray as xr
import pandas as pd
import scipy # for interpolation

torch.set_printoptions(sci_mode = False)

# Data

- Phase ice velocity data is 6.9 GB originally and 4.9 GB in .pt subset.
- BecMachine v3 is 8.5 GB originally.

## Load BedMachine

In [3]:
bedmachine = xr.open_dataset("/home/kim/data/nsidc/BedMachineAntarctica-v3.nc")

In [4]:
# bound to domain of phase_velocity (inclusive of boundries)
bedmachine = bedmachine.sel(x = slice(-2800000, 2800000), y = slice(2800000, -2800000))

bedmachine

<xarray.Dataset>
Dimensions:    (x: 11201, y: 11201)
Coordinates:
  * x          (x) int32 -2800000 -2799500 -2799000 ... 2799000 2799500 2800000
  * y          (y) int32 2800000 2799500 2799000 ... -2799000 -2799500 -2800000
Data variables:
    mapping    |S1 ...
    mask       (y, x) int8 ...
    firn       (y, x) float32 ...
    surface    (y, x) float32 ...
    thickness  (y, x) float32 ...
    bed        (y, x) float32 ...
    errbed     (y, x) float32 ...
    source     (y, x) int8 ...
    dataid     (y, x) int8 ...
    geoid      (y, x) int16 ...
Attributes: (12/17)
    Conventions:                 CF-1.7
    Title:                       BedMachine Antarctica
    Author:                      Mathieu Morlighem
    version:                     03-Jun-2022 (v3.4)
    nx:                          13333.0
    ny:                          13333.0
    ...                          ...
    ymax:                        3333000
    spacing:                     500
    no_data:                     -9999.0
    license:                     No restrictions on access or use
    Data_citation:               Morlighem M. et al., (2019), Deep glacial tr...
    Notes:                       Data processed at the Department of Earth Sy...

## Create tensor

In [5]:
bedmachine_dims = len(bedmachine.x.values)

bedmachine_tensor = torch.cat((torch.tensor(bedmachine.bed.values).unsqueeze(0),
                               torch.tensor(bedmachine.surface.values).unsqueeze(0),
                               torch.tensor(bedmachine.thickness.values).unsqueeze(0),
                               torch.tensor(bedmachine.mask.values).unsqueeze(0),
                               torch.tensor(bedmachine.firn.values).unsqueeze(0),
                               torch.tensor(bedmachine.errbed.values).unsqueeze(0),
                               # YX
                               torch.tensor(bedmachine.coords["y"].values).unsqueeze(-1).repeat(1, bedmachine_dims).unsqueeze(0),
                               torch.tensor(bedmachine.coords["x"].values).repeat(bedmachine_dims, 1).unsqueeze(0)),
                               dim = 0)

## Load Phase velocity

In [6]:
phase_velocity = xr.open_dataset("/home/kim/data/nsidc/antarctic_ice_vel_phase_map_v01.nc")

In [7]:
# Reassuring it's consistent
phase_velocity = phase_velocity.sel(x = slice(-2800000, 2800000), y = slice(2800000, -2800000))

phase_velocity

<xarray.Dataset>
Dimensions:       (x: 12445, y: 12445)
Coordinates:
  * x             (x) float64 -2.8e+06 -2.8e+06 -2.799e+06 ... 2.799e+06 2.8e+06
  * y             (y) float64 2.8e+06 2.8e+06 2.799e+06 ... -2.799e+06 -2.8e+06
    lat           (y, x) float64 ...
    lon           (y, x) float64 ...
Data variables:
    coord_system  |S1 ...
    VX            (y, x) float32 ...
    VY            (y, x) float32 ...
    STDX          (y, x) float32 ...
    STDY          (y, x) float32 ...
    ERRX          (y, x) float32 ...
    ERRY          (y, x) float32 ...
    CNT           (y, x) int32 ...
    SOURCE        (y, x) int8 ...
Attributes: (12/27)
    Conventions:               CF-1.6
    Metadata_Conventions:      CF-1.6, Unidata Dataset Discovery v1.0, GDS v2.0
    standard_name_vocabulary:  CF Standard Name Table (v22, 12 February 2013)
    id:                        v_mix.v8Jul2019.nc
    title:                     MEaSURES Antarctica Ice Velocity Map 450m spacing
    product_version:            
    ...                        ...
    time_coverage_start:       1995-01-01
    time_coverage_end:         2016-12-31
    project:                   NASA/MEaSUREs
    creator_name:              J. Mouginot
    comment:                    
    license:                   No restrictions on access or use.

In [8]:
"""
phase_velocity_dims = len(phase_velocity.x.values)

# YX order
phase_velocity_tensor = torch.cat((torch.tensor(phase_velocity.data_vars["VY"].values).unsqueeze(0),
                              torch.tensor(phase_velocity.data_vars["VX"].values).unsqueeze(0),
                              torch.tensor(phase_velocity.coords["y"].values).unsqueeze(-1).repeat(1, phase_velocity_dims).unsqueeze(0),
                              torch.tensor(phase_velocity.coords["x"].values).repeat(phase_velocity_dims, 1).unsqueeze(0)),
                              dim = 0)
"""

'\nphase_velocity_dims = len(phase_velocity.x.values)\n\n# YX order\nphase_velocity_tensor = torch.cat((torch.tensor(phase_velocity.data_vars["VY"].values).unsqueeze(0),\n                              torch.tensor(phase_velocity.data_vars["VX"].values).unsqueeze(0),\n                              torch.tensor(phase_velocity.coords["y"].values).unsqueeze(-1).repeat(1, phase_velocity_dims).unsqueeze(0),\n                              torch.tensor(phase_velocity.coords["x"].values).repeat(phase_velocity_dims, 1).unsqueeze(0)),\n                              dim = 0)\n'

## Interpolation attempt

In [9]:
phase_velocity_tensor[0].unsqueeze(0).unsqueeze(0).shape

NameError: name 'phase_velocity_tensor' is not defined

In [ ]:
# Y 
y_min_midpoint = torch.min(phase_velocity_tensor[2, :, 0])
y_max_midpoint = torch.max(phase_velocity_tensor[2, :, 0])

print("-1 corresponds to :", y_min_midpoint - 225)
print("1 corresponds to :", y_max_midpoint + 225)
print("2 units span :", y_max_midpoint - y_min_midpoint + 450)

# Loosing some 
2.0/(y_max_midpoint - y_min_midpoint + 450)
2.0/(y_max_midpoint - y_min_midpoint + 450)*225
2.0/(y_max_midpoint - y_min_midpoint + 450)*450

-1 corresponds to : tensor(-2800025., dtype=torch.float64)
1 corresponds to : tensor(2800225., dtype=torch.float64)
2 units span : tensor(5600250., dtype=torch.float64)


tensor(0.0002, dtype=torch.float64)

In [ ]:
phase_velocity_tensor[3, 5, :]

tensor([-2800000., -2799550., -2799100.,  ...,  2798900.,  2799350.,
         2799800.], dtype=torch.float64)

In [ ]:
x_min_midpoint = torch.min(phase_velocity_tensor[3, 0, :])
x_max_midpoint = torch.max(phase_velocity_tensor[3, 0, :])

In [ ]:
torch.tensor(2000000000.0, dtype = torch.double)/(y_max_midpoint - y_min_midpoint + 450)

tensor(357.1269, dtype=torch.float64)

In [ ]:
# y_max - 225 is -1 and y_max + 225 is 1
((y_max_midpoint + 225) - (y_min_midpoint - 225))/450

tensor(12445., dtype=torch.float64)

In [ ]:
y_min_midpoint

tensor(-2799800., dtype=torch.float64)

In [ ]:
y_max_midpoint

tensor(2800000., dtype=torch.float64)

In [ ]:
# start and end are included
dy = torch.linspace(start = 0, 
                   end = (y_max_midpoint - y_min_midpoint), 
                   steps = phase_velocity_dims)
dy = (((dy + 225) / (y_max_midpoint - y_min_midpoint + 450)) * 2) -1

dx = torch.linspace(start = 0, 
                   end = (x_max_midpoint - x_min_midpoint), 
                   steps = phase_velocity_dims)
dx = (((dx + 225) / (x_max_midpoint - x_min_midpoint + 450)) * 2) -1

meshx, meshy = torch.meshgrid((dx, dy), indexing = "xy") # create mesh
target_grid = torch.stack((meshx, meshy), 2) # x,y order
target_grid = target_grid.unsqueeze(0) # N dimension

In [ ]:
out = torch.nn.functional.grid_sample(
                # VY velocity
                input = phase_velocity_tensor[0].unsqueeze(0).unsqueeze(0).float(),
                # grid construction
                grid = target_grid.float(), 
                mode = 'bilinear', 
                # mode = 'nearest', 
                # mode = 'bicubic', 
                # Padding mode is not so relevant as we only use inner values anyways
                padding_mode = 'border', 
                align_corners = False)

In [ ]:
torch.sum(torch.isclose(out, phase_velocity_tensor[0].unsqueeze(0).unsqueeze(0).float(), equal_nan = True))

tensor(148177379)

In [ ]:
phase_velocity_tensor[0].unsqueeze(0).unsqueeze(0)[0, 0, 6000:6008, 6000:6008]

tensor([[10.2599, 10.1900, 10.1133, 10.0270,  9.9312,  9.8301,  9.7308,  9.6353],
        [10.2161, 10.1414, 10.0531,  9.9557,  9.8513,  9.7442,  9.6416,  9.5450],
        [10.1571, 10.0742,  9.9773,  9.8722,  9.7639,  9.6560,  9.5513,  9.4534],
        [10.0916,  9.9997,  9.8954,  9.7858,  9.6761,  9.5669,  9.4612,  9.3623],
        [10.0234,  9.9240,  9.8161,  9.7041,  9.5937,  9.4855,  9.3834,  9.2864],
        [ 9.9571,  9.8534,  9.7440,  9.6305,  9.5186,  9.4118,  9.3180,  9.2282],
        [ 9.8934,  9.7869,  9.6756,  9.5613,  9.4513,  9.3533,  9.2724,  9.1891],
        [ 9.8337,  9.7245,  9.6124,  9.5002,  9.3983,  9.3089,  9.2303,  9.1500]],
       dtype=torch.float64)

In [ ]:
out[0, 0, 6000:6008, 6000:6008]

tensor([[10.2599, 10.1900, 10.1133, 10.0270,  9.9312,  9.8301,  9.7308,  9.6353],
        [10.2161, 10.1414, 10.0531,  9.9557,  9.8513,  9.7442,  9.6416,  9.5450],
        [10.1571, 10.0742,  9.9773,  9.8722,  9.7639,  9.6560,  9.5513,  9.4534],
        [10.0916,  9.9997,  9.8954,  9.7858,  9.6761,  9.5669,  9.4612,  9.3623],
        [10.0234,  9.9240,  9.8161,  9.7041,  9.5937,  9.4855,  9.3834,  9.2864],
        [ 9.9571,  9.8534,  9.7440,  9.6305,  9.5186,  9.4118,  9.3180,  9.2282],
        [ 9.8934,  9.7869,  9.6756,  9.5613,  9.4513,  9.3533,  9.2724,  9.1891],
        [ 9.8337,  9.7245,  9.6124,  9.5002,  9.3983,  9.3089,  9.2303,  9.1500]])

In [ ]:
bm_y_min = torch.min(bedmachine_tensor[6, :, 0])
print("Bedmachine y min:", bm_y_min.item())
bm_y_max = torch.max(bedmachine_tensor[6, :, 0])
print("Bedmachine y max:", bm_y_max.item())

bm_midpoint_range = bm_y_max - bm_y_min
print("Bedmachine range of midpoints:", bm_midpoint_range.item())
bm_range = bm_midpoint_range + 500
print("Bedmachine full range of grid cells (outer boundaries):", bm_range.item())
print(bm_range)

bedmachine_dims
d = (torch.linspace(start = 0,
               end = bm_midpoint_range,
               steps = bedmachine_dims
               ) + (500.0/2))/bm_range


Bedmachine y min: -2800000.0
Bedmachine y max: 2800000.0
Bedmachine range of midpoints: 5600000.0
Bedmachine full range of grid cells (outer boundaries): 5600500.0
tensor(5600500.)


# xarray interpolation

In [10]:
# Bilinear interpolation
VX_interpol = phase_velocity.VX.interp(x = bedmachine.coords["x"].values, y = bedmachine.coords["y"].values, method = "linear")
VY_interpol = phase_velocity.VY.interp(x = bedmachine.coords["x"].values, y = bedmachine.coords["y"].values, method = "linear")

In [ ]:
"""
# Test: Bilinear interpolation around South Pole
test = phase_velocity.sel(x = slice(-500, 500), y = slice(500, -500))
test.VX[:, 1]

VX_interpol[5600, 5600]

(1/4.5)*(1/4.5) + (1/4.5)*(3.5/4.5) + (1/4.5)*(3.5/4.5) + (3.5/4.5)*(3.5/4.5)
(1/4.5)*(1/4.5)*(-5.333) + (1/4.5)*(3.5/4.5)*(-5.338) + (1/4.5)*(3.5/4.5)*(-5.4140) + (3.5/4.5)*(3.5/4.5)*(-5.4147)
# torch.mean(torch.tensor([-5.333, -5.338, -5.4140, -5.4147]))
"""

'\n# Test: Bilinear interpolation around South Pole\ntest = phase_velocity.sel(x = slice(-500, 500), y = slice(500, -500))\ntest.VX[:, 1]\n\nVX_interpol[5600, 5600]\n\n(1/4.5)*(1/4.5) + (1/4.5)*(3.5/4.5) + (1/4.5)*(3.5/4.5) + (3.5/4.5)*(3.5/4.5)\n(1/4.5)*(1/4.5)*(-5.333) + (1/4.5)*(3.5/4.5)*(-5.338) + (1/4.5)*(3.5/4.5)*(-5.4140) + (3.5/4.5)*(3.5/4.5)*(-5.4147)\n# torch.mean(torch.tensor([-5.333, -5.338, -5.4140, -5.4147]))\n'

In [11]:
tensor = torch.cat((bedmachine_tensor[[0, 1, 2, 3]],
                    torch.tensor(VY_interpol.values).unsqueeze(0),
                    torch.tensor(VX_interpol.values).unsqueeze(0),
                    bedmachine_tensor[[6, 7]]), 
                    dim = 0)

In [ ]:
torch.corrcoef(tensor[0:6].reshape(6, -1))

tensor([[1.0000, 0.6926, 0.6314, 0.7664,    nan,    nan],
        [0.6926, 1.0000, 0.9334, 0.6989,    nan,    nan],
        [0.6314, 0.9334, 1.0000, 0.7382,    nan,    nan],
        [0.7664, 0.6989, 0.7382, 1.0000,    nan,    nan],
        [   nan,    nan,    nan,    nan,    nan,    nan],
        [   nan,    nan,    nan,    nan,    nan,    nan]], dtype=torch.float64)

# Combined tensor

- [0, :, : ] bed elevation
- [1, :, : ] surface elevation
- [2, :, : ] thickess elevation
- [3, :, : ] mask
- [4, :, : ] VY
- [5, :, : ] VX
- [6, :, : ] Y (high to low)
- [7, :, : ] X (low to high)

In [13]:
# save file locally
torch.save(tensor, '/home/kim/data/bedmachine_phase_velocity_tensor.pt')

In [51]:
# Lets look at the bed
out[0, :, :].shape

# Terrain Roughness Index

torch.Size([4001, 4001])

In [37]:
torch.sum(tensor[4,:, :].isnan())/(11201*11201)
torch.sum(tensor[5,:, :].isnan())/(11201*11201)

torch.sum(tensor[4, tensor[3,:,:] != 0.0].isnan()) # still 278161 are 0
tensor[4, tensor[3,:,:] != 0.0].isnan().shape

torch.sum(tensor[5, tensor[3,:,:] != 0.0].isnan()) # 278161
tensor[3, tensor[3,:,:] != 0.0]

tensor([3., 3., 3.,  ..., 2., 2., 2.], dtype=torch.float64)